In [ ]:
# %load_ext autoreload
# %autoreload 2
import fedbo.visualization.results as results
import pandas as pd
import scikit_posthocs as sp
from critdd import Diagram
from pathlib import Path
import matplotlib.pyplot as plt

style_path = results.STYLE_PATH
plt.style.use(style_path)

In [ ]:
import os
ENTITY = os.environ["WANDB_ENTITY"]
METRICS = ["avg/simple_regret"]
CONFIG_KEYS = ["trainer._target_", "model._target_"]

#### Homogeneous Taks Benchmark

In [ ]:
# Load Project Data
projects = ["ackley1d", "ackley5d", "ackley10d", "combination1d","combination5d", "combination10d", "branin", "hartmann6d", "cartpole"]
df_list = [results.load_wandb_data(ENTITY, p, METRICS, CONFIG_KEYS) for p in projects]

In [ ]:
print(df_list[0])

Comparison with Federated Baselines

In [ ]:
models_to_plot = None  # ["FL-MLP", "FL-VBLL", "DP-FTS-DE", "FMTBO", "FMTBO_2"]
fig = results.plot_objective_data(df_list, projects, models_to_plot)
fig.savefig("comp_fbo.pdf")

Comparison with Local Baselines

In [ ]:
models_to_plot = ["FL-MLP", "FL-VBLL", "L-GP", "L-MLP", "L-VBLL"]
fig = results.plot_objective_data(df_list, ["ackley1d", "ackley5d", "ackley10d"], models_to_plot, force_nrows=2)
fig.savefig("comp_local_ackley.pdf")

In [ ]:
models_to_plot = ["FL-MLP", "FL-VBLL", "L-GP", "L-MLP", "L-VBLL"]
fig = results.plot_objective_data(df_list, projects, models_to_plot)
fig.savefig("comp_local_full.pdf")

Comparison with the Centralized Baseline

In [ ]:
models_to_plot = ["FL-MLP", "FL-VBLL", "C-MLP", "C-VBLL"]
fig = results.plot_objective_data(df_list, ["ackley1d", "ackley5d", "ackley10d"], models_to_plot, force_nrows=2)
fig.savefig("comp_centralized_ackley.pdf")

In [ ]:
models_to_plot = ["FL-MLP", "FL-VBLL", "C-MLP", "C-VBLL"]
fig = results.plot_objective_data(df_list, projects, models_to_plot)
fig.savefig("comp_centralized_full.pdf")

Warm Starting

In [ ]:
ws_projects = [
    "ackley1d-warm_start",
    "ackley5d-warm_start",
    "ackley10d-warm_start",
    "combination1d-warm_start",
    "combination5d-warm_start",
    "combination10d-warm_start",
    "branin-warm_start",
    "hartmann6d-warm_start",
    "cartpole-warm_start",
]

metrics_agent_0 = ["simple_regret/agent_0"]
ws_df_list_agent_0 = [results.load_wandb_data(ENTITY, p, metrics_agent_0, CONFIG_KEYS) for p in ws_projects]

metrics_agent_10 = ["simple_regret/agent_10"]
ws_df_list_agent_10 = [results.load_wandb_data(ENTITY, p, metrics_agent_10, CONFIG_KEYS) for p in ws_projects]

MLP Only

In [ ]:
models_to_plot = None  # ["FL-MLP", "FMTBO", "DP-FTS-DE"]
fig = results.plot_warm_start_data(ws_df_list_agent_0, ws_df_list_agent_10, ws_projects, models_to_plot, force_nrows=4)
fig.savefig("warm_start_mlp.pdf")

VBLL Only

In [ ]:
models_to_plot = ["FL-VBLL", "FMTBO", "DP-FTS-DE"]
fig = results.plot_warm_start_data(ws_df_list_agent_0, ws_df_list_agent_10, ws_projects, models_to_plot, force_nrows=4)
fig.savefig("warm_start_vbll.pdf")

#### Heterogeneous Task Benchmark

10D

In [ ]:
num_agents = 18
manytask_data_keys = [f"simple_regret/agent_{i}" for i in range(18)]
manytask_df_list = [results.load_wandb_data(ENTITY, "manytask10d", [dk], CONFIG_KEYS) for dk in manytask_data_keys]

In [ ]:
manytask_models = ["FL-VBLL", "C-VBLL", "L-VBLL", "L-GP", "FMTBO", "DP-FTS-DE"]
fig = results.plot_manytask_data(manytask_df_list[:6], manytask_models)
fig.savefig("manytask10d_partial.pdf")

In [ ]:
fig = results.plot_manytask_data(manytask_df_list[6:], manytask_models, limit_y=True)
fig.savefig("manytask10d_full.pdf")

50D

In [ ]:
manytask50d_df_list = [results.load_wandb_data(ENTITY, "manytask50d", [dk], CONFIG_KEYS) for dk in manytask_data_keys]

In [ ]:
manytask_models = ["FL-MLH", "FL-MLP", "C-MLP", "L-GP", "FMTBO", "DP-FTS-DE"]
fig = results.plot_manytask_data(manytask50d_df_list[:6], manytask_models, limit_y=False)
fig.savefig("manytask50d_partial.pdf")

In [ ]:
manytask_models = ["FL-VBLL", "FL-MLP", "FL-MLH", "L-GP", "FMTBO", "DP-FTS-DE"]
fig = results.plot_manytask_data(manytask50d_df_list[6:], manytask_models, limit_y=False)
fig.savefig("manytask50d_full.pdf")

In [ ]:
manytask_models = ["FL-VBLL", "FL-MLP", "FL-MLH", "L-GP", "FMTBO", "DP-FTS-DE"]
fig = results.plot_manytask_data(manytask50d_df_list[9:], manytask_models, limit_y=False)
fig.savefig("manytask50d_full2.pdf")

#### Ablation Study

In [ ]:
projects = ["ackley1d", "ackley5d", "ackley10d", "combination1d","combination5d", "combination10d", "branin", "hartmann6d", "cartpole"]
ablation_cfg_keys = ["trainer._target_", "model._target_", "num_agents", "model.hidden_features", "trainer.num_rounds", "model.num_hidden_layers"]

df_list_obj_data_params = [results.load_wandb_data(ENTITY, p, METRICS, ablation_cfg_keys) for p in projects]

In [ ]:
df_list_obj_data_params_filtered = [df[(df['trainer._target_'] == 'fedbo.training.FederatedTrainer.hydra_init') & (df['model._target_'] == 'fedbo.models.VBLL.hydra_init')] for df in df_list_obj_data_params]

In [ ]:
df_list_obj_data_params_filtered = [df[(df['trainer._target_'] == 'fedbo.training.FederatedTrainer.hydra_init') & (df['model._target_'] == 'fedbo.models.MLP.hydra_init')] for df in df_list_obj_data_params]

##### Num Agents

In [ ]:
projects = ["ackley1d", "ackley5d", "ackley10d", "combination1d","combination5d", "combination10d", "branin", "hartmann6d", "cartpole"]
num_agents_cfg_keys = ["trainer._target_", "model._target_", "num_agents"]

suffix = "num_agents"
df_list_num_agents = [results.load_wandb_data(ENTITY, f"{p}-{suffix}", METRICS, num_agents_cfg_keys) for p in projects]

In [ ]:
df_list_num_agents_merged: list[pd.DataFrame]  = []
for df_params, df_num_agents in zip(df_list_obj_data_params_filtered, df_list_num_agents):
    df_list_num_agents_merged.append(pd.concat([df_params, df_num_agents]))

In [ ]:
models_to_plot = ["FL-VBLL"]
fig = results.plot_num_agents_data(df_list_num_agents_merged, projects, models_to_plot)
fig.savefig("num_agents.pdf")

In [ ]:
models_to_plot = ["FL-MLP"]
fig = results.plot_num_agents_data(df_list_num_agents_merged, projects, models_to_plot)
fig.savefig("num_agents_mlp.pdf")

##### Num Aggregations

In [ ]:
projects = ["ackley1d", "ackley5d", "ackley10d", "combination1d","combination5d", "combination10d", "branin", "hartmann6d", "cartpole"]
config_keys = ["trainer._target_", "model._target_", "trainer.num_rounds"]

suffix = "num_aggregations"
df_list_num_aggr = [results.load_wandb_data(ENTITY, f"{p}-{suffix}", METRICS, config_keys) for p in projects]

In [ ]:
df_list_num_aggr_merged: list[pd.DataFrame]  = []
for df_params, df_num_aggr in zip(df_list_obj_data_params_filtered, df_list_num_aggr):
    df_list_num_aggr_merged.append(pd.concat([df_params, df_num_aggr]))

In [ ]:
models_to_plot = ["FL-VBLL"]
fig = results.plot_num_aggregations_data(df_list_num_aggr_merged, projects, models_to_plot)
fig.savefig("num_aggregations.pdf")

In [ ]:
models_to_plot = ["FL-MLP"]
fig = results.plot_num_aggregations_data(df_list_num_aggr_merged, projects, models_to_plot)
fig.savefig("num_aggregations_mlp.pdf")

##### Num Features

In [ ]:
projects = ["ackley1d", "ackley5d", "ackley10d", "combination1d","combination5d", "combination10d", "branin", "hartmann6d", "cartpole"]
config_keys = ["trainer._target_", "model._target_", "model.hidden_features"]

suffix = "num_hidden_features"
df_list_num_feat = [results.load_wandb_data(ENTITY, f"{p}-{suffix}", METRICS, config_keys) for p in projects]

In [ ]:
df_list_params_feat_filtered = [df[((df['trainer._target_'] == 'fedbo.training.FederatedTrainer.hydra_init') & 
                                       (df['model._target_'] == 'fedbo.models.VBLL.hydra_init')) |
                                       ((df['trainer._target_'] == 'fedbo.training.LocalTrainer.hydra_init') & 
                                       (df['model._target_'] == 'fedbo.models.VBLL.hydra_init'))] for df in df_list_obj_data_params]
df_list_num_feat_merged: list[pd.DataFrame]  = []
for df_params, df_num_feat in zip(df_list_params_feat_filtered, df_list_num_feat):
    df_list_num_feat_merged.append(pd.concat([df_params, df_num_feat]))

In [ ]:
df_list_params_feat_filtered = [df[((df['trainer._target_'] == 'fedbo.training.FederatedTrainer.hydra_init') & 
                                       (df['model._target_'] == 'fedbo.models.MLP.hydra_init')) |
                                       ((df['trainer._target_'] == 'fedbo.training.LocalTrainer.hydra_init') & 
                                       (df['model._target_'] == 'fedbo.models.MLP.hydra_init'))] for df in df_list_obj_data_params]
df_list_num_feat_merged: list[pd.DataFrame]  = []
for df_params, df_num_feat in zip(df_list_params_feat_filtered, df_list_num_feat):
    df_list_num_feat_merged.append(pd.concat([df_params, df_num_feat]))

In [ ]:
models_to_plot = ["FL-VBLL"]
fig = results.plot_num_hidden_features_data(df_list_num_feat_merged, projects, models_to_plot)
fig.savefig("num_hidden_features.pdf")

In [ ]:
models_to_plot = ["FL-MLP"]
fig = results.plot_num_hidden_features_data(df_list_num_feat_merged, projects, models_to_plot)
fig.savefig("num_hidden_features_mlp.pdf")

##### Num Backbone Layers

In [ ]:
projects = ["ackley1d", "ackley5d", "ackley10d", "combination1d","combination5d", "combination10d", "branin", "hartmann6d", "cartpole"]
config_keys = ["trainer._target_", "model._target_", "model.num_hidden_layers"]

suffix = "num_hidden_layers"
df_list_num_layers = [results.load_wandb_data(ENTITY, f"{p}-{suffix}", METRICS, config_keys) for p in projects]

In [ ]:
df_list_num_layers_merged: list[pd.DataFrame]  = []
for df_params, df_num_layers in zip(df_list_obj_data_params_filtered, df_list_num_layers):
    df_list_num_layers_merged.append(pd.concat([df_params, df_num_layers]))

In [ ]:
models_to_plot = ["FL-VBLL"]
fig = results.plot_num_layers_data(df_list_num_layers_merged, projects, models_to_plot)
fig.savefig("num_layers.pdf")

In [ ]:
models_to_plot = ["FL-MLP"]
fig = results.plot_num_layers_data(df_list_num_layers_merged, projects, models_to_plot)
fig.savefig("num_layers_mlp.pdf")

##### Num Head Layers

In [ ]:
projects = ["ackley1d", "ackley5d", "ackley10d", "combination1d","combination5d", "combination10d", "branin", "hartmann6d", "cartpole"]
config_keys = ["trainer._target_", "model._target_", "model.num_head_layers"]

suffix = "mlh_vbll"
df_list_num_head_layers = [results.load_wandb_data(ENTITY, f"{p}-{suffix}", METRICS, config_keys) for p in projects]

In [ ]:
fig = results.plot_num_head_layers_data(df_list_num_head_layers, projects)
fig.savefig("num_head_layers.pdf")

In [ ]:
projects = ["ackley1d", "ackley5d", "ackley10d", "combination1d","combination5d", "combination10d", "branin", "hartmann6d", "cartpole"]
config_keys = ["trainer._target_", "model._target_", "model.num_head_layers"]

suffix = "mlh_mlp"
df_list_num_head_layers = [results.load_wandb_data(ENTITY, f"{p}-{suffix}", METRICS, config_keys) for p in projects]

In [ ]:
fig = results.plot_num_head_layers_data(df_list_num_head_layers, projects)
fig.savefig("num_head_layers_mlp.pdf")

#### Mean Rank Diagram

In [ ]:
import os
WANDB_ENTITY = os.environ["WANDB_ENTITY"]
PROJECT_LIST = [
    "ackley1d",
    "ackley5d",
    "ackley10d",
    "branin",
    "hartmann6d",
    "combination1d",
    "combination5d",
    "combination10d",
    "cartpole",
]

print("--- Starting Final Regret Data Extraction ---")
later_iter_data = results.get_final_regret_data(WANDB_ENTITY, PROJECT_LIST)
rank_df = pd.DataFrame(later_iter_data)

In [ ]:
# Rank each row (lower value = better rank)
ranks = rank_df.rank(axis=1, method="average", ascending=True)

# Compute mean rank per algorithm
mean_ranks = ranks.mean().sort_values()

# Plot mean rank
plt.figure(figsize=(8, 4))
plt.hlines(y=mean_ranks.index, xmin=min(mean_ranks)-0.2, xmax=max(mean_ranks)+0.2, color='lightgray')
plt.plot(mean_ranks.values, mean_ranks.index, 'o', color='blue', markersize=8)

plt.title("Simple Mean Rank Plot of Algorithms")
plt.xlabel("Mean Rank (lower = better)")
plt.gca().invert_xaxis()  # Show best on the left
plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.show()

print(mean_ranks)

In [ ]:
%matplotlib inline
avg_ranks_series = rank_df.rank(axis=1).mean(axis=0)
p_values = sp.posthoc_nemenyi_friedman(rank_df)

fig, ax = plt.subplots(figsize=(10, 4))

sp.critical_difference_diagram(
    ranks=avg_ranks_series,
    sig_matrix=p_values,
    ax=ax
)

plt.title('Critical Difference Diagram (Nemenyi Test)')
plt.tight_layout()

In [ ]:
# Create a CD diagram from the Pandas DataFrame
diagram = Diagram(
    rank_df.to_numpy(),
    treatment_names = rank_df.columns,
    maximize_outcome = False
)

# Average ranks and 
print(diagram.average_ranks)
# Groups of statistically indistinguishable treatments
print(diagram.get_groups(alpha=.05, adjustment="holm"))

# Export the diagram to a file
diagram.to_file(
    "mean_rank.tex",
    alpha = .05,
    adjustment = "holm",
    reverse_x = True,
)

In [ ]:
%matplotlib inline

mt_10d_rank = {
    "FL-VBLL": [
        0.905, 73.3, 4.03, 73.2, 9.28, 2600.0, 77.5, 176.0, 4.29, 15.4,
        4.19, 2.59, 15.0, 76.5, 0.955, 2.05, 72.3, 2670.0
    ],
    "C-VBLL": [
        0.768, 64.9, 3.5, 66.5, 8.87, 2610.0, 66.7, 84.3, 3.6, 13.3,
        3.42, 2.25, 13.6, 66.9, 0.963, 1.72, 66.0, 2630.0
    ],
    "L-VBLL": [
        1.01, 116.0, 7.58, 136.0, 7.61, 2670.0, 118.0, 150.0, 8.06, 40.2,
        7.82, 4.85, 43.1, 136.0, 0.986, 4.28, 128.0, 2680.0
    ],
    "L-GP": [
        1.04, 166.0, 11.6, 176.0, 12.4, 2530.0, 173.0, 246.0, 11.9, 189.0,
        11.2, 8.75, 216.0, 195.0, 1.06, 6.6, 186.0, 2520.0
    ],
    "FMTBO": [
        1.0, 108.0, 6.57, 123.0, 7.7, 2220.0, 106.0, 178.0, 7.2, 964000.0,
        7.29, 5.78, 1390000.0, 133.0, 1.02, 4.42, 114.0, 2100.0
    ],
    "DP-FTS-DE": [
        1.36, 371.0, 15.4, 333.0, 18.9, 2720.0, 432.0, 10200.0, 15.3,
        125000000.0, 15.9, 11.7, 141000000.0, 438.0, 1.41, 9.96, 431.0, 2870.0
    ],
    "IAF-FBO": [
        0.814, 62.8, 3.59, 65.2, 15.6, 2730.0, 67.0, 55.0, 3.74, 7780.0,
        3.85, 3.28, 8240.0, 63.3, 0.876, 1.85, 65.9, 2830.0
    ],
    "FD-EMD": [
        0.571, 69.6, 3.69, 63.6, 19.0, 2590.0, 64.5, 45.5, 4.0, 24500.0,
        3.95, 7.94, 23000.0, 68.1, 1.0, 6.9, 63.8, 2610.0
    ]
}

mt10d_rank_df = pd.DataFrame(mt_10d_rank)


avg_ranks_series = mt10d_rank_df.rank(axis=1).mean(axis=0)
p_values = sp.posthoc_nemenyi_friedman(mt10d_rank_df)

fig, ax = plt.subplots(figsize=(10, 4))

sp.critical_difference_diagram(
    ranks=avg_ranks_series,
    sig_matrix=p_values,
    ax=ax
)

plt.title('Critical Difference Diagram (Nemenyi Test)')
plt.tight_layout()

print(avg_ranks_series)

In [ ]:
%matplotlib inline

mt_50d_rank = {
    "FL-MLP": [
        3.01e1, 2.99e4, 2.14e1, 3.00e4, 2.14e1, 1.77e4, 3.02e4, 1.15e5, 
        2.13e1, 1.30e4, 2.13e1, 7.78e1, 1.34e4, 2.94e4, 3.05e1, 7.88e1, 
        2.97e4, 1.75e4
    ],
    "FL-MLH": [
        2.13e1, 2.09e4, 2.12e1, 2.10e4, 2.14e1, 1.68e4, 2.16e4, 8.49e4, 
        2.12e1, 8.11e3, 2.12e1, 7.83e1, 8.11e3, 2.02e4, 2.15e1, 7.55e1, 
        2.04e4, 1.70e4
    ],
    "C-MLP": [
        2.36e1, 2.29e4, 2.13e1, 2.30e4, 2.14e1, 1.78e4, 2.29e4, 9.38e4, 
        2.13e1, 1.09e4, 2.13e1, 7.80e1, 1.18e4, 2.26e4, 2.45e1, 7.64e1, 
        2.32e4, 1.76e4
    ],
    "L-MLP": [
        3.10e1, 3.06e4, 2.14e1, 3.10e4, 2.14e1, 1.80e4, 3.08e4, 1.23e5, 
        2.14e1, 1.41e4, 2.14e1, 7.91e1, 1.43e4, 2.99e4, 3.18e1, 7.89e1, 
        3.00e4, 1.77e4
    ],
    "L-GP": [
        2.89e1, 2.77e4, 2.13e1, 2.80e4, 2.14e1, 1.75e4, 2.80e4, 1.08e5, 
        2.14e1, 1.16e4, 2.14e1, 7.83e1, 1.22e4, 2.72e4, 2.79e1, 7.75e1, 
        2.77e4, 1.72e4
    ],
    "FMTBO": [
        2.73e1, 2.62e4, 2.12e1, 2.65e4, 2.09e1, 1.72e4, 2.65e4, 1.12e5, 
        2.12e1, 3.59e9, 2.12e1, 9.04e1, 3.65e9, 2.66e4, 2.65e1, 8.29e1, 
        2.67e4, 1.73e4
    ],
    "DP-FTS-DE": [
        2.81e1, 2.36e4, 2.13e1, 2.37e4, 2.10e1, 1.79e4, 2.30e4, 1.03e5, 
        2.12e1, 2.60e9, 2.13e1, 1.06e2, 2.60e9, 2.32e4, 2.80e1, 8.96e1, 
        2.53e4, 1.77e4
    ],
    "IAF-FBO": [
        1.35e0, 8.15e2, 9.85e0, 8.09e2, 2.14e1, 1.79e4, 8.19e2, 8.31e3, 
        1.04e1, 6.99e5, 9.45e0, 2.40e1, 7.05e5, 8.08e2, 2.09e0, 2.43e1, 
        7.87e2, 1.77e4
    ],
    "FD-EMD": [
        1.25e0, 1.16e3, 1.18e1, 1.18e3, 2.10e1, 1.74e4, 1.13e3, 1.03e4, 
        1.20e1, 2.60e6, 1.20e1, 8.63e1, 2.67e6, 1.13e3, 2.41e0, 8.18e1, 
        1.13e3, 1.75e4
    ]
}

mt50d_rank_df = pd.DataFrame(mt_50d_rank)


avg_ranks_series = mt50d_rank_df.rank(axis=1).mean(axis=0)
p_values = sp.posthoc_nemenyi_friedman(mt50d_rank_df)

fig, ax = plt.subplots(figsize=(10, 4))

sp.critical_difference_diagram(
    ranks=avg_ranks_series,
    sig_matrix=p_values,
    ax=ax
)

plt.title('Critical Difference Diagram (Nemenyi Test)')
plt.tight_layout()

print(avg_ranks_series)

In [ ]:
# Create a CD diagram from the Pandas DataFrame
diagram = Diagram(
    mt10d_rank_df.to_numpy(),
    treatment_names = mt10d_rank_df.columns,
    maximize_outcome = False
)

# Average ranks and 
print(diagram.average_ranks)
# Groups of statistically indistinguishable treatments
print(diagram.get_groups(alpha=.1, adjustment="holm"))

# Export the diagram to a file
diagram.to_file(
    "mean_rank_manytask10d.tex",
    alpha = .1,
    adjustment = "holm",
    reverse_x = True,
)